# K-Sentinel: 45 checks and open training

Train an open residual CNN on split-preserving synthetic complex MRI data, reproduce the released failure mechanism, and inspect the evidence. **Synthetic research only.** Forty-five passing software/scientific contract checks do not establish clinical safety, natural pathology detection or state-of-the-art performance.

Author: [Vinsy Moyya](https://orcid.org/0009-0003-0042-3556). [Code and paper](https://github.com/vinsymoyya/k-sentinel). Code/model: MIT. Generated data: CC0-1.0.

This notebook runs on CPU with internet disabled. Add the public K-Sentinel dataset and model variation if you fork it. Every original training, calibration and test identity is preserved. The original unreviewed preprint remains a v0.1.0 experimental record; this notebook is the v0.2.0 reproducibility workflow.


In [ ]:
from pathlib import Path
import sys, os, zipfile, json, subprocess, shutil
destination = Path('/kaggle/working')
root = destination / 'k-sentinel-0.2.0'
archives = list(Path('/kaggle/input').rglob('k-sentinel-0.2.0-source.bundle'))
if not archives:
    archives = list(Path('/kaggle/input').rglob('k-sentinel-0.2.0-source.zip'))
if len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as bundle:
        for item in bundle.namelist():
            if not (destination / item).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Archive member escapes working directory')
        bundle.extractall(destination)
else:
    # Kaggle expands uploaded model ZIP files into directories.
    sources = [p.parents[1] for p in Path('/kaggle/input').rglob('run_validation.py')
               if p.parent.name == 'scripts' and (p.parents[1] / 'ksentinel').is_dir()]
    if len(sources) != 1:
        raise ValueError('Attach exactly one K-Sentinel model source bundle')
    shutil.copytree(sources[0], root, dirs_exist_ok=True)
os.chdir(root)
sys.path.insert(0, str(root))
import numpy as np, torch
print({'numpy': np.__version__, 'torch': str(torch.__version__), 'source': str(root)})


## 1. Run every validation case

No skips are counted as passes. The runner refuses a suite other than the documented 45 cases and writes JSON plus JUnit evidence.

In [ ]:
subprocess.run([sys.executable, 'scripts/run_validation.py', '--output', '/kaggle/working/validation'], check=True)
validation = json.loads(Path('/kaggle/working/validation/report.json').read_text())
assert validation['passed'] == 45 and validation['failed'] == 0 and validation['skipped'] == 0
print({key: validation[key] for key in ('total', 'passed', 'failed', 'skipped')})


## 2. Inspect the downloadable dataset

Each shard stores complex truth, initially acquired columns/values, independent audit columns/values, identities and noise scale. Keep audit values out of reconstruction. Ground truth is used only as a training target or for final evaluation.

In [ ]:
from ksentinel.dataset import load_partition
candidates = [p for p in Path('/kaggle/input').rglob('manifest.json') if (p.parent / 'seed11-train.npz').exists()]
if len(candidates) != 1:
    raise ValueError('Attach exactly one K-Sentinel synthetic MRI dataset')
data_dir = candidates[0].parent
manifest = json.loads(candidates[0].read_text())
assert manifest['total_objects'] == 3840
print({'objects': manifest['total_objects'], 'shards': len(manifest['files']), 'license': manifest['license']})
sample = load_partition(data_dir, 'test', 11, limit=1)
import matplotlib.pyplot as plt
plt.figure(figsize=(4, 4))
plt.imshow(abs(sample['truth'][0]), cmap='gray', vmin=0, vmax=1)
plt.title('Synthetic test phantom; no patient data')
plt.axis('off'); plt.show()


## 3. Train with fixed settings, then evaluate

Eight epochs, Adam at 0.001, batch size 32, CPU. Three seeds. Calibration and test subjects never train the network; test scores never select a checkpoint. Set QUICK=True only for a small smoke run; it does not reproduce the full experiment.

In [ ]:
from ksentinel.training import train_dataset
QUICK = False
reports = []
for seed in ([11] if QUICK else [11, 29, 47]):
    reports.append(train_dataset(data_dir, '/kaggle/working/training', seed=seed,
                                 epochs=1 if QUICK else 8, limit=16 if QUICK else None))
print([{key: r[key] for key in ('test_count', 'mean_cnn_nrmse', 'mean_zero_fill_nrmse')} for r in reports])


## 4. Compare reconstruction error

Lower complex NRMSE is better. These are results for the current hosted run, not clinical endpoints. Inspect natural reconstruction error before interpreting an added-error audit.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for report in reports:
    axes[0].plot(range(1, len(report['training_mse']) + 1), report['training_mse'], label=f"seed {report['config']['seed']}")
axes[0].set(xlabel='Epoch', ylabel='Training channel MSE', title='Training history')
axes[0].legend()
positions = np.arange(len(reports))
axes[1].bar(positions - .17, [r['mean_zero_fill_nrmse'] for r in reports], width=.34, label='Zero fill')
axes[1].bar(positions + .17, [r['mean_cnn_nrmse'] for r in reports], width=.34, label='CNN')
axes[1].set(xticks=positions, xticklabels=[str(r['config']['seed']) for r in reports], xlabel='Seed', ylabel='Mean test NRMSE', title='Held-out synthetic test subjects')
axes[1].legend(); plt.tight_layout(); plt.show()


## 5. Load and use the open checkpoint

Only the initial measured coefficients enter reconstruction. Values from the unacquired grid are ignored. The audit remains a separate check and a negative flag does not certify correct anatomy.

In [ ]:
from ksentinel.model_io import load_model, reconstruct
from ksentinel.dataset import observed_kspace
checkpoint = Path('/kaggle/working/training/cnn_seed11.pt')
model = load_model(checkpoint, expected_sha256=reports[0]['checkpoint_sha256'])
measured, initial_mask = observed_kspace(sample, 0)
candidate = reconstruct(model, measured, initial_mask)
plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1); plt.imshow(abs(sample['truth'][0]), cmap='gray', vmin=0, vmax=1); plt.title('Synthetic truth'); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(abs(candidate), cmap='gray', vmin=0, vmax=1); plt.title('Trained CNN reconstruction'); plt.axis('off')
plt.tight_layout(); plt.show()
Path('/kaggle/working/run-summary.json').write_text(json.dumps({'quick': QUICK, 'validation': {'passed': validation['passed'], 'total': validation['total']}, 'training_reports': [str(p) for p in Path('/kaggle/working/training').glob('*.json')], 'scope': 'Synthetic methods workflow; no clinical validation'}, indent=2))


## Reuse and contribution

Download the generated checkpoints and JSON/JUnit records from notebook output. Fork this notebook for independent reproduction; preserve split identities and report failures. The attached model contains the original open weights and a source archive. The Kaggle model page documents licensing and intended use. Downloads, forks and votes reflect actual community use; no growth outcome is promised.